In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# --- Parameters ---
nPaths = 100       # number of Brownian bridges
nPoints = 26       # 0–25 inclusive
x = np.arange(nPoints)

# --- Simulate Brownian bridges ---
dW = np.random.randn(nPaths, nPoints - 1)
W = np.cumsum(dW, axis=1)
W = np.column_stack((np.zeros(nPaths), W))  # start at 0

t = np.linspace(0, 1, nPoints)
bridges = W - np.outer(W[:, -1], t)

# --- Compute maximum absolute bridge amplitude ---
maxBridgeVal = np.max(np.abs(bridges))

# --- Define smooth, realistic CUSUM shapes ---
shapePos = np.sin(np.pi * t)   # gradual rise and fall
shapeNeg = -np.sin(np.pi * t)  # mirror dip

# Add small random noise
rng = np.random.default_rng(42)
noiseLevel = 0.15
cusumPos = shapePos + rng.normal(0, noiseLevel, nPoints)
cusumNeg = shapeNeg + rng.normal(0, noiseLevel, nPoints)

# Anchor both at 0 start/end
cusumPos -= (t / t[-1]) * cusumPos[-1]
cusumNeg -= (t / t[-1]) * cusumNeg[-1]
cusumPos -= cusumPos[0]
cusumNeg -= cusumNeg[0]

# Scale to match bridge amplitude
cusumPos *= (maxBridgeVal / np.max(np.abs(cusumPos))) * 0.92
cusumNeg *= (maxBridgeVal / np.max(np.abs(cusumNeg))) * 0.9

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 6))
colors = cm.viridis(np.linspace(0, 1, nPaths))
for i in range(nPaths):
    ax.plot(x, bridges[i], color=colors[i], alpha=0.35, linewidth=0.8)

ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.plot(x, cusumPos, color='green', linewidth=2.5, label='CUSUM statistic (+ve deviation)')
ax.plot(x, cusumNeg, color='blue', linewidth=2.5, label='CUSUM statistic (–ve deviation)')

# --- Style ---
ax.set_title('Brownian Bridges and Two-Sided Simulated CUSUM Statistics', fontsize=16)
ax.set_xlabel('Trial (0–25)', fontsize=14)
ax.set_ylabel('Value', fontsize=14)
ax.set_xticks(np.arange(0, nPoints))
ax.tick_params(axis='both', which='major', labelsize=12)

# Remove top/right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.legend(fontsize=12)
fig.tight_layout()
plt.show()
